In [1]:
import warnings

import pandas as pd
import numpy as np
from sklearn.exceptions import ConvergenceWarning
pd.set_option('display.max_colwidth', None)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings(
    "ignore",
    message="Bins whose width are too small (i.e., <= 1e-8)*",
)

np.random.seed(94357)

# Primer conjunto de datos (nodos 0..20000)
Este conjunto corresponde al subgrafo construido con los nodos identificados del 0 al 20000 (rango fijo y reproducible).
Se usa como muestra manejable del grafo completo para extraer relaciones y combinar atributos.
El objetivo es reducir el volumen sin perder estructura, facilitando el analisis y el modelado.

## Primer modelo: Naive Bayes

Aplicaremos Naive Bayes como linea base porque es rapido y funciona bien con variables independientes o casi independientes. El modelo estima la probabilidad de cada clase dado un conjunto de caracteristicas y elige la clase con mayor probabilidad. Aunque la suposicion de independencia es fuerte, suele dar resultados competitivos y nos sirve para comparar con modelos mas complejos.

In [2]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [3]:
from sklearn.preprocessing import KBinsDiscretizer

# Cargar datos y preparar X/y
ruta = "generated_CSV/features_relaciones_20k.csv"

df = pd.read_csv(ruta)
columnas_excluir = ["created_at", "updated_at", "numeric_id"]
columnas_relacionales = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

y = df["affiliate"].copy()
# Definición de varios conjuntos de atributos diferentes
conjuntos_atributos = {
    "DF 1 - todos los atributos": df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 2 - sin atributos relacionales": df.drop(columns=columnas_excluir + ["affiliate"] + columnas_relacionales, errors="ignore").copy(),
    "DF 3 - solo atributos relacionales": df[columnas_relacionales].copy(),
    "DF 4 - originales + pageRank + closeness_centrality": df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 5 - atributos seleccionados por chi2": df[["views", "life_time", "triangles"]].copy(),
}
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["pageRank"] = df["pageRank"]
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["closeness_centrality"] = df["closeness_centrality"]

# Elegir el conjunto por defecto, para cambiar el conjunto usado, se ha de cambiar la clave usada
X = conjuntos_atributos["DF 1 - todos los atributos"].copy()
atributos_continuos = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

discretizador = KBinsDiscretizer(
    n_bins=40,
    encode="ordinal",
    strategy="kmeans",
)
X_disc = X.copy()
#Con el DF 2 hay que comentar esta parte porque no tenemos atributos continuos
atr_contenidos = list(set(atributos_continuos) & set(X_disc.columns))
if atr_contenidos:
    X_disc[atr_contenidos] = discretizador.fit_transform(
        X_disc[atr_contenidos]
    )

# Codificar variables categoricas (incluye numericas como categorias si vienen como object)
encoder = OrdinalEncoder()
X_enc = encoder.fit_transform(X_disc)

# Codificar la variable objetivo
label_enc = LabelEncoder()
y_enc = label_enc.fit_transform(y)

In [4]:
# Particion train/test
atr_train, atr_test, obj_train, obj_test = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=94357, stratify=y_enc
)

In [5]:
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB())])
rejilla_de_hiperparámetros = {
    'naive_Bayes__alpha': range(1, 10),
    'naive_Bayes__force_alpha': [True, False],
}


In [ ]:
búsqueda_en_rejilla = GridSearchCV(tubería_NB,
                                   rejilla_de_hiperparámetros,
                                   scoring='accuracy',
                                   cv=10)
búsqueda_en_rejilla.fit(atr_train, obj_train)

In [ ]:
búsqueda_en_rejilla.best_estimator_

In [ ]:
búsqueda_en_rejilla.best_score_

In [ ]:
# Evaluacion
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB(alpha=4, force_alpha=True))])
tubería_NB.fit(atr_train, obj_train)
tubería_NB.score(atr_test, obj_test)

In [ ]:
predicciones_NB = tubería_NB.predict(atr_test)
acc_test_nb = accuracy_score(obj_test, predicciones_NB)

print('=== Mejor configuración Naive Bayes ===')
print('Mejores hiperparámetros:', búsqueda_en_rejilla.best_params_)
print(f"Mejor accuracy en validación cruzada: {búsqueda_en_rejilla.best_score_:.4f}")
print(f"Mejor accuracy en test: {acc_test_nb:.4f}")
print('Informe de clasificación:')
print(classification_report(
    obj_test,
    predicciones_NB,
    target_names=[str(c) for c in label_enc.classes_],
))

In [ ]:
#OBTENCIÓN DE LAS MEJORES FEATURES CON CHI2
from sklearn.feature_selection import SelectKBest, chi2

X_cat = X_enc.astype(int)

chi2_features = SelectKBest(chi2, k=3)
X_kbest_features = chi2_features.fit_transform(X_cat, y_enc)

# Variables reducidas
print("Original feature number:", X_cat.shape[1])
print("Reduced feature number:", X_kbest_features.shape[1])


feature_names = X.columns.tolist()
selected_mask = chi2_features.get_support() # Devuelve un array booleano indicando qué columnas han sido seleccionadas
selected_features = [name for name, keep in zip(feature_names, selected_mask) if keep]
print("Selected features:", selected_features)

## Segundo modelo: CART (Classification and Regression Trees)

CART es un algoritmo de aprendizaje supervisado que construye un árbol de decisión mediante divisiones binarias recursivas. En cada nodo, el algoritmo selecciona el atributo y el umbral que mejor separan las clases, maximizando la pureza de los subconjuntos resultantes (usando métricas como el índice Gini o la entropía).

A diferencia de Naive Bayes, CART no asume independencia entre las características y puede capturar relaciones no lineales e interacciones complejas entre variables. Además, es interpretable visualmente, ya que el árbol resultante muestra explícitamente las reglas de decisión.

Para evitar el sobreajuste, aplicaremos poda del árbol limitando su profundidad máxima (`max_depth`) y el número mínimo de muestras necesarias para dividir un nodo (`min_samples_split`). Estos hiperparámetros se optimizarán mediante búsqueda en rejilla con validación cruzada.

In [12]:
from sklearn.tree import DecisionTreeClassifier

In [13]:
clasificador_CART = DecisionTreeClassifier(random_state=94357)
rejilla_de_hiperparámetros_CART = {
    'max_depth': range(5, 50), # profundidad máxima del árbol para evitar el overfitting
    'min_samples_split': range(5, 25, 5) # número de muestras mínimo para dividir un nodo y prevenir el overfitting
}

In [ ]:
búsqueda_en_rejilla_CART = GridSearchCV(clasificador_CART,
                                        rejilla_de_hiperparámetros_CART,
                                        scoring='accuracy',
                                        cv=10)
búsqueda_en_rejilla_CART.fit(atr_train, obj_train)

In [ ]:
búsqueda_en_rejilla_CART.best_params_

In [ ]:
búsqueda_en_rejilla_CART.best_score_

In [ ]:
mejor_CART = búsqueda_en_rejilla_CART.best_estimator_
mejor_CART.fit(atr_train, obj_train)
print("Accuracy en entrenamiento:", mejor_CART.score(atr_train, obj_train))
print("Accuracy en prueba:", mejor_CART.score(atr_test, obj_test))

In [ ]:
predicciones_CART = mejor_CART.predict(atr_test)
print(classification_report(obj_test, predicciones_CART, target_names=[str(c) for c in label_enc.classes_]))

## Tercer modelo: KNN

Para este tercer modelo probamos varias combinaciones de atributos, igual que en Naive Bayes, para ver con cuál obtenemos mejor precisión. El bloque carga el CSV ya preparado, prueba distintos subconjuntos de variables y luego entrena y ajusta un KNN para cada configuración. Además, este modelo depende mucho de la escala de las variables y de la distancia entre ejemplos, así que comparar varias versiones del conjunto de datos ayuda a ver qué atributos aportan más información y cuáles solo añaden ruido.


In [19]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import KBinsDiscretizer, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [20]:
# Repetimos el flujo de preparación para poder comparar distintos conjuntos de atributos.
ruta = "generated_CSV/features_relaciones_20k.csv"

df_knn = pd.read_csv(ruta)
columnas_excluir = ["created_at", "updated_at", "numeric_id"]
columnas_relacionales = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

y_knn = df_knn["affiliate"].copy()
conjuntos_atributos = {
    "DF 1 - todos los atributos": df_knn.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 2 - sin atributos relacionales": df_knn.drop(columns=columnas_excluir + ["affiliate"] + columnas_relacionales, errors="ignore").copy(),
    "DF 3 - solo atributos relacionales": df_knn[columnas_relacionales].copy(),
    "DF 4 - originales + pageRank + closeness_centrality": df_knn.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 5 - atributos seleccionados por chi2": df_knn[["views", "life_time", "triangles"]].copy(),
}
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["pageRank"] = df_knn["pageRank"]
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["closeness_centrality"] = df_knn["closeness_centrality"]

atributos_continuos = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]
label_enc_knn = LabelEncoder()
y_enc_knn = label_enc_knn.fit_transform(y_knn)

parametros_busqueda_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['minkowski', 'manhattan'],
}

def preparar_conjunto_knn(X_knn):
    discretizador_knn = KBinsDiscretizer(
        n_bins=40,
        encode='ordinal',
        strategy='kmeans',
    )
    X_disc_knn = X_knn.copy()
    atr_contenidos = list(set(atributos_continuos) & set(X_disc_knn.columns))
    if atr_contenidos:
        X_disc_knn[atr_contenidos] = discretizador_knn.fit_transform(X_disc_knn[atr_contenidos])

    encoder_knn = OrdinalEncoder()
    X_enc_knn = encoder_knn.fit_transform(X_disc_knn)
    
    atr_train_knn, atr_test_knn, obj_train_knn, obj_test_knn = train_test_split(
        X_enc_knn,
        y_enc_knn,
        test_size=0.2,
        random_state=94357,
        stratify=y_enc_knn,
    )
    return atr_train_knn, atr_test_knn, obj_train_knn, obj_test_knn

In [ ]:
resultados_knn = []
mejor_resultado_knn = None

for nombre_conjunto, X_knn in conjuntos_atributos.items():
    print(f'\n=== {nombre_conjunto} ===')
    atr_train_knn, atr_test_knn, obj_train_knn, obj_test_knn = preparar_conjunto_knn(X_knn)

    tuberia_knn = Pipeline([
        ('escalador', StandardScaler()),
        ('knn', KNeighborsClassifier()),
    ])

    busqueda_knn = GridSearchCV(
        tuberia_knn,
        parametros_busqueda_knn,
        scoring='accuracy',
        cv=10,
        n_jobs=-1,
    )
    busqueda_knn.fit(atr_train_knn, obj_train_knn)

    predicciones_knn = busqueda_knn.best_estimator_.predict(atr_test_knn)
    accuracy_test = accuracy_score(obj_test_knn, predicciones_knn)

    resultado = {
        'conjunto': nombre_conjunto,
        'best_params': busqueda_knn.best_params_,
        'cv_accuracy': busqueda_knn.best_score_,
        'test_accuracy': accuracy_test,
        'modelo': busqueda_knn.best_estimator_,
        'X_test': atr_test_knn,
        'y_test': obj_test_knn,
        'y_pred': predicciones_knn,
        'label_enc': label_enc_knn,
    }
    resultados_knn.append(resultado)

    if mejor_resultado_knn is None or resultado['test_accuracy'] > mejor_resultado_knn['test_accuracy']:
        mejor_resultado_knn = resultado

In [ ]:
resumen_knn = pd.DataFrame([
    {
        'conjunto': r['conjunto'],
        'cv_accuracy': r['cv_accuracy'],
        'test_accuracy': r['test_accuracy'],
        'best_params': r['best_params'],
    }
    for r in resultados_knn
]).sort_values('test_accuracy', ascending=False)

display(resumen_knn)

print('=== Mejor configuración KNN ===')
print(mejor_resultado_knn['conjunto'])
print('Mejores hiperparámetros:', mejor_resultado_knn['best_params'])
print(f"Mejor accuracy en validación cruzada: {mejor_resultado_knn['cv_accuracy']:.4f}")
print(f"Mejor accuracy en test: {mejor_resultado_knn['test_accuracy']:.4f}")
print('Informe de clasificación:')
print(classification_report(
    mejor_resultado_knn['y_test'],
    mejor_resultado_knn['y_pred'],
    target_names=[str(c) for c in mejor_resultado_knn['label_enc'].classes_],
))

## Cuarto modelo: Redes neuronales

Las redes neuronales son modelos de aprendizaje inspirados en el cerebro que aprenden patrones a partir de datos mediante capas de neuronas artificiales.

**Idea clave**
- Una neurona aplica una combinacion lineal de entradas y luego una funcion de activacion para producir una salida.

**Arquitectura basica**
- Capa de entrada: recibe las caracteristicas.
- Capas ocultas: extraen representaciones intermedias.
- Capa de salida: produce la prediccion.

**Entrenamiento**
- Se usa descenso de gradiente y retropropagacion para ajustar los pesos.
- La funcion de perdida mide el error (por ejemplo, entropia cruzada en clasificacion).

**Ventajas y limites**
- Ventajas: capturan relaciones no lineales y escalan bien con datos.
- Limites: requieren mas datos, ajuste de hiperparametros y mayor costo computacional.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import keras

print("Keras backend:", keras.backend.backend())

In [24]:
from keras.utils import set_random_seed
set_random_seed(94357)

In [25]:
from keras import Sequential, Input
from keras.layers import Dense, Normalization, Dropout
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

In [26]:
normalizador = Normalization()
normalizador.adapt(atr_train)

In [27]:
#Sigmoid = clasificación binaria (affiliate 0/1)
#relu = y >= 0
#linear = regresión (neg/pos)

red = Sequential()
red.add(Input(shape=(atr_train.shape[1],)))
red.add(normalizador)
red.add(Dense(64, activation='relu'))
red.add(Dropout(0.3))
red.add(Dense(32, activation='relu'))
red.add(Dropout(0.3))
red.add(Dense(16, activation='relu'))
red.add(Dropout(0.2))
red.add(Dense(1, activation='sigmoid'))

In [28]:
red.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
 )

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

In [ ]:
historial = red.fit(
    atr_train, obj_train,
    batch_size=256,
    epochs=300,
    validation_split=0.2, # El 0.2 representa 
    callbacks=[early_stopping], # El entrenamiento se detendrá si la pérdida de validación no mejora durante 20 épocas consecutivas, y restaurará los pesos del modelo al estado con la mejor pérdida de validación
    verbose=1
)

In [ ]:
red.evaluate(atr_test, obj_test)

# Segundo conjunto de datos (nodos 20000..40000)
Este conjunto corresponde al subgrafo construido con los nodos identificados del 20000 al 40000 (rango fijo y reproducible).
Se usa como muestra manejable del grafo completo para extraer relaciones y combinar atributos.
El objetivo es reducir el volumen sin perder estructura, facilitando el analisis y el modelado.

## Primer modelo: Naive Bayes

Aplicaremos Naive Bayes como linea base porque es rapido y funciona bien con variables independientes o casi independientes. El modelo estima la probabilidad de cada clase dado un conjunto de caracteristicas y elige la clase con mayor probabilidad. Aunque la suposicion de independencia es fuerte, suele dar resultados competitivos y nos sirve para comparar con modelos mas complejos.

In [1]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# Cargar datos y preparar X/y
ruta = "generated_CSV/features_relaciones_20k_to_40k.csv"

df = pd.read_csv(ruta)
columnas_excluir = ["created_at", "updated_at", "numeric_id"]
columnas_relacionales = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

y = df["affiliate"].copy()
# Definición de varios conjuntos de atributos para probar
conjuntos_atributos = {
    "DF 1 - todos los atributos": df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 2 - sin atributos relacionales": df.drop(columns=columnas_excluir + ["affiliate"] + columnas_relacionales, errors="ignore").copy(),
    "DF 3 - solo atributos relacionales": df[columnas_relacionales].copy(),
    "DF 4 - originales + pageRank + closeness_centrality": df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 5 - atributos seleccionados por chi2": df[["views", "life_time", "triangles"]].copy(),
}
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["pageRank"] = df["pageRank"]
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["closeness_centrality"] = df["closeness_centrality"]

# Elegir el conjunto por defecto, para cambiar el conjunto usado, se ha de cambiar la clave usada
X = conjuntos_atributos["DF 2 - sin atributos relacionales"].copy()

# Discretizar atributos continuos antes de codificar
atributos_continuos = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

discretizador = KBinsDiscretizer(
    n_bins=40,
    encode="ordinal",
    strategy="kmeans",
)
X_disc = X.copy()
atr_contenidos = list(set(atributos_continuos) & set(X_disc.columns))
if atr_contenidos:
    X_disc[atr_contenidos] = discretizador.fit_transform(
        X_disc[atr_contenidos]
    )

# Codificar variables categoricas
encoder = OrdinalEncoder()
X_enc = encoder.fit_transform(X_disc)

# Codificar la variable objetivo
label_enc = LabelEncoder()
y_enc = label_enc.fit_transform(y)

In [ ]:
# Particion train/test
atr_train, atr_test, obj_train, obj_test = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=94357, stratify=y_enc
)

In [ ]:
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB())])
rejilla_de_hiperparámetros = {
    'naive_Bayes__alpha': range(1, 10),
    'naive_Bayes__force_alpha': [True, False],
}


In [ ]:
búsqueda_en_rejilla = GridSearchCV(tubería_NB,
                                   rejilla_de_hiperparámetros,
                                   scoring='accuracy',
                                   cv=10)
búsqueda_en_rejilla.fit(atr_train, obj_train)

In [ ]:
búsqueda_en_rejilla.best_estimator_

In [ ]:
búsqueda_en_rejilla.best_score_

In [ ]:
# Evaluación
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB(alpha=2, force_alpha=True))])
tubería_NB.fit(atr_train, obj_train)
tubería_NB.score(atr_test, obj_test)

In [ ]:
predicciones_NB = tubería_NB.predict(atr_test)
acc_test_nb = accuracy_score(obj_test, predicciones_NB)

print('=== Mejor configuración Naive Bayes ===')
print('Mejores hiperparámetros:', búsqueda_en_rejilla.best_params_)
print(f"Mejor accuracy en validación cruzada: {búsqueda_en_rejilla.best_score_:.4f}")
print(f"Mejor accuracy en test: {acc_test_nb:.4f}")
print('Informe de clasificación:')
print(classification_report(
    obj_test,
    predicciones_NB,
    target_names=[str(c) for c in label_enc.classes_],
))

In [ ]:
#OBTENCIÓN DE LAS MEJORES FEATURES CON CHI2
from sklearn.feature_selection import SelectKBest, chi2

X_cat = X_enc.astype(int)

chi2_features = SelectKBest(chi2, k=3)
X_kbest_features = chi2_features.fit_transform(X_cat, y_enc)

# Variables reducidas
print("Original feature number:", X_cat.shape[1])
print("Reduced feature number:", X_kbest_features.shape[1])


feature_names = X.columns.tolist()
selected_mask = chi2_features.get_support() # Devuelve un array booleano indicando qué columnas han sido seleccionadas
selected_features = [name for name, keep in zip(feature_names, selected_mask) if keep]
print("Selected features:", selected_features)

## Segundo modelo: CART (Classification and Regression Trees)

CART es un algoritmo de aprendizaje supervisado que construye un arbol de decision mediante divisiones binarias recursivas. En cada nodo, el algoritmo selecciona el atributo y el umbral que mejor separan las clases, maximizando la pureza de los subconjuntos resultantes (usando metricas como el indice Gini o la entropia).

A diferencia de Naive Bayes, CART no asume independencia entre las caracteristicas y puede capturar relaciones no lineales e interacciones complejas entre variables. Ademas, es interpretable visualmente, ya que el arbol resultante muestra explicitamente las reglas de decision.

Para evitar el sobreajuste, aplicaremos poda del arbol limitando su profundidad maxima (`max_depth`) y el numero minimo de muestras necesarias para dividir un nodo (`min_samples_split`). Estos hiperparametros se optimizaran mediante busqueda en rejilla con validacion cruzada.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
clasificador_CART = DecisionTreeClassifier(random_state=94357)
rejilla_de_hiperparámetros_CART = {
    'max_depth': range(5, 50), # profundidad máxima del árbol para evitar el overfitting
    'min_samples_split': range(5, 25, 5) # número de muestras mínimo para dividir un nodo y prevenir el overfitting
}

In [ ]:
búsqueda_en_rejilla_CART = GridSearchCV(clasificador_CART,
                                        rejilla_de_hiperparámetros_CART,
                                        scoring='accuracy',
                                        cv=10)
búsqueda_en_rejilla_CART.fit(atr_train, obj_train)

In [ ]:
búsqueda_en_rejilla_CART.best_params_

In [ ]:
búsqueda_en_rejilla_CART.best_score_

In [ ]:
mejor_CART = búsqueda_en_rejilla_CART.best_estimator_
mejor_CART.fit(atr_train, obj_train)
print("Accuracy en entrenamiento:", mejor_CART.score(atr_train, obj_train))
print("Accuracy en prueba:", mejor_CART.score(atr_test, obj_test))

In [ ]:
predicciones_CART = mejor_CART.predict(atr_test)
print(classification_report(obj_test, predicciones_CART, target_names=[str(c) for c in label_enc.classes_]))

## Tercer modelo: KNN

Para este tercer modelo probamos varias combinaciones de atributos, igual que en Naive Bayes, para ver con cuál obtenemos mejor precisión. El bloque carga el CSV ya preparado, prueba distintos subconjuntos de variables y luego entrena y ajusta un KNN para cada configuración. Además, este modelo depende mucho de la escala de las variables y de la distancia entre ejemplos, así que comparar varias versiones del conjunto de datos ayuda a ver qué atributos aportan más información y cuáles solo añaden ruido.

In [29]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import KBinsDiscretizer, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [30]:
# Repetimos el flujo de preparación para poder comparar distintos conjuntos de atributos.
ruta = "generated_CSV/features_relaciones_20k_to_40k.csv"

df_knn = pd.read_csv(ruta)
columnas_excluir = ["created_at", "updated_at", "numeric_id"]
columnas_relacionales = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

y_knn = df_knn["affiliate"].copy()
conjuntos_atributos = {
    "DF 1 - todos los atributos": df_knn.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 2 - sin atributos relacionales": df_knn.drop(columns=columnas_excluir + ["affiliate"] + columnas_relacionales, errors="ignore").copy(),
    "DF 3 - solo atributos relacionales": df_knn[columnas_relacionales].copy(),
    "DF 4 - originales + pageRank + closeness_centrality": df_knn.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy(),
    "DF 5 - atributos seleccionados por chi2": df_knn[["views", "life_time", "triangles"]].copy(),
}
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["pageRank"] = df_knn["pageRank"]
conjuntos_atributos["DF 4 - originales + pageRank + closeness_centrality"]["closeness_centrality"] = df_knn["closeness_centrality"]

atributos_continuos = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]
label_enc_knn = LabelEncoder()
y_enc_knn = label_enc_knn.fit_transform(y_knn)

parametros_busqueda_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['minkowski', 'manhattan'],
}

def preparar_conjunto_knn(X_knn):
    discretizador_knn = KBinsDiscretizer(
        n_bins=40,
        encode='ordinal',
        strategy='kmeans',
    )
    X_disc_knn = X_knn.copy()
    atr_contenidos = list(set(atributos_continuos) & set(X_disc_knn.columns))
    if atr_contenidos:
        X_disc_knn[atr_contenidos] = discretizador_knn.fit_transform(X_disc_knn[atr_contenidos])

    encoder_knn = OrdinalEncoder()
    X_enc_knn = encoder_knn.fit_transform(X_disc_knn)
    
    atr_train_knn, atr_test_knn, obj_train_knn, obj_test_knn = train_test_split(
        X_enc_knn,
        y_enc_knn,
        test_size=0.2,
        random_state=94357,
        stratify=y_enc_knn,
    )
    return atr_train_knn, atr_test_knn, obj_train_knn, obj_test_knn

In [ ]:
resultados_knn = []
mejor_resultado_knn = None

for nombre_conjunto, X_knn in conjuntos_atributos.items():
    print(f'\n=== {nombre_conjunto} ===')
    atr_train_knn, atr_test_knn, obj_train_knn, obj_test_knn = preparar_conjunto_knn(X_knn)

    tuberia_knn = Pipeline([
        ('escalador', StandardScaler()),
        ('knn', KNeighborsClassifier()),
    ])

    busqueda_knn = GridSearchCV(
        tuberia_knn,
        parametros_busqueda_knn,
        scoring='accuracy',
        cv=10,
        n_jobs=-1,
    )
    busqueda_knn.fit(atr_train_knn, obj_train_knn)

    predicciones_knn = busqueda_knn.best_estimator_.predict(atr_test_knn)
    accuracy_test = accuracy_score(obj_test_knn, predicciones_knn)

    resultado = {
        'conjunto': nombre_conjunto,
        'best_params': busqueda_knn.best_params_,
        'cv_accuracy': busqueda_knn.best_score_,
        'test_accuracy': accuracy_test,
        'modelo': busqueda_knn.best_estimator_,
        'X_test': atr_test_knn,
        'y_test': obj_test_knn,
        'y_pred': predicciones_knn,
        'label_enc': label_enc_knn,
    }
    resultados_knn.append(resultado)

    if mejor_resultado_knn is None or resultado['test_accuracy'] > mejor_resultado_knn['test_accuracy']:
        mejor_resultado_knn = resultado

In [ ]:
resumen_knn = pd.DataFrame([
    {
        'conjunto': r['conjunto'],
        'cv_accuracy': r['cv_accuracy'],
        'test_accuracy': r['test_accuracy'],
        'best_params': r['best_params'],
    }
    for r in resultados_knn
]).sort_values('test_accuracy', ascending=False)

display(resumen_knn)

print('=== Mejor configuración KNN ===')
print(mejor_resultado_knn['conjunto'])
print('Mejores hiperparámetros:', mejor_resultado_knn['best_params'])
print(f"Mejor accuracy en validación cruzada: {mejor_resultado_knn['cv_accuracy']:.4f}")
print(f"Mejor accuracy en test: {mejor_resultado_knn['test_accuracy']:.4f}")
print('Informe de clasificación:')
print(classification_report(
    mejor_resultado_knn['y_test'],
    mejor_resultado_knn['y_pred'],
    target_names=[str(c) for c in mejor_resultado_knn['label_enc'].classes_],
))

## Cuarto modelo: Redes neuronales

Las redes neuronales son modelos de aprendizaje inspirados en el cerebro que aprenden patrones a partir de datos mediante capas de neuronas artificiales.

**Idea clave**
- Una neurona aplica una combinacion lineal de entradas y luego una funcion de activacion para producir una salida.

**Arquitectura basica**
- Capa de entrada: recibe las caracteristicas.
- Capas ocultas: extraen representaciones intermedias.
- Capa de salida: produce la prediccion.

**Entrenamiento**
- Se usa descenso de gradiente y retropropagacion para ajustar los pesos.
- La funcion de perdida mide el error (por ejemplo, entropia cruzada en clasificacion).

**Ventajas y limites**
- Ventajas: capturan relaciones no lineales y escalan bien con datos.
- Limites: requieren mas datos, ajuste de hiperparametros y mayor costo computacional.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import keras

print("Keras backend:", keras.backend.backend())

In [ ]:
from keras.utils import set_random_seed
set_random_seed(94357)

In [ ]:
from keras import Sequential, Input
from keras.layers import Dense, Normalization, Dropout
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

In [ ]:
normalizador_2 = Normalization()
normalizador_2.adapt(atr_train)

In [ ]:
red_2 = Sequential()
red_2.add(Input(shape=(atr_train.shape[1],)))
red_2.add(normalizador_2)
red_2.add(Dense(64, activation='relu'))
red_2.add(Dropout(0.3))
red_2.add(Dense(32, activation='relu'))
red_2.add(Dropout(0.3))
red_2.add(Dense(16, activation='relu'))
red_2.add(Dropout(0.2))
red_2.add(Dense(1, activation='sigmoid'))

In [ ]:
red_2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
 )

early_stopping_2 = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

In [ ]:
historial_2 = red_2.fit(
    atr_train, obj_train,
    batch_size=256,
    epochs=300,
    validation_split=0.2,
    callbacks=[early_stopping_2],
    verbose=1
)

In [ ]:
red_2.evaluate(atr_test, obj_test)